# Modeling

### Load Data

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/dataset_final.csv")
df = df.sort_values("date")

df.head()

,date,ihsg,nilai_tukar,suku_bunga,inflasi,pdb
0,2015-01-02,5242.77,12542.5,7.75,-0.24,2.15804
1,2015-01-05,5219.99,12627.5,7.75,-0.24,2.15804
2,2015-01-06,5169.06,12657.5,7.75,-0.24,2.15804
3,2015-01-07,5207.12,12738.5,7.75,-0.24,2.15804
4,2015-01-08,5211.83,12680.0,7.75,-0.24,2.15804


### Split Data Train & Test

In [3]:
from src.modeling import train_test_split_time_series

train, test = train_test_split_time_series(df.drop(columns=["date"]))

### Scaling

In [4]:
from src.modeling import scale_data

train_scaled, test_scaled, scaler = scale_data(train, test)

### Windowing

In [5]:
from src.modeling import create_sequences

window_size = 30

X_train, y_train = create_sequences(train_scaled, window_size)
X_test, y_test = create_sequences(test_scaled, window_size)

### Build LSTM Model

In [6]:
from src.modeling import build_lstm_model

model = build_lstm_model((X_train.shape[1], X_train.shape[2]))
model.summary()

d:\project_skripsi\.venv\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,985 (70.25 KB)

 Trainable params: 17,985 (70.25 KB)

 Non-trainable params: 0 (0.00 B)

### Training

In [7]:
history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 5s 28ms/step - loss: 0.0168 - val_loss: 0.0034
Epoch 2/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 0.0020 - val_loss: 0.0018
Epoch 3/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - loss: 0.0016 - val_loss: 0.0014
Epoch 4/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 0.0013 - val_loss: 0.0011
Epoch 5/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - loss: 0.0012 - val_loss: 0.0013
Epoch 6/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0011 - val_loss: 0.0011
Epoch 7/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 25ms/step - loss: 0.0011 - val_loss: 9.9701e-04
Epoch 8/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - loss: 0.0010 - val_loss: 7.2712e-04
Epoch 9/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - loss: 9.0678e-04 - val_loss: 0.0013
Epoch 10/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 9.4052e-04 - val_loss: 0.0011
Epoch 11/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - loss: 8.9248e-04 - val_loss: 9.2467e-04
Epoch 12/50
60/60 ━━━━━━━━━━━━━━━━━━━━ 

### Evaluation

In [8]:
from src.modeling import evaluate_model

y_pred = model.predict(X_test)

rmse, mape, r2 = evaluate_model(y_test, y_pred)

print("RMSE:", rmse)
print("MAPE:", mape)
print("R2:", r2)

15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step
RMSE: 0.01706241292740092
MAPE: 0.014339268019646696
R2: 0.961570193132226


save history

In [ ]:
import numpy as np

np.save("../model/model_eks1/y_test.npy", y_test)
np.save("../model/model_eks1/y_pred.npy", y_pred)

import json
with open("../model/model_eks1/history.json", "w") as f:
    json.dump(history.history, f)

In [13]:
model.save("../model/model_eks1/lstm_model.keras")

In [12]:
import joblib
joblib.dump(scaler, "../model/model_eks1/scaler.save")


['../model/model_eks1/scaler.save']